# Notebook 18 — ISLES2024 preprocessed tensor integrity audit (WU-18)

**Run in Google Colab with a CPU runtime: Runtime → Run all.** No GPU or model training is involved.

This workflow inspects the **stored, already-normalized nnU-Net v2 3D full-resolution `.b2nd` tensors** used for Dataset505 (149 patients × NCCT/Tmax/CBF/CBV/MTT). It compares patient-level channel summaries with Notebook 17's **raw staged NIfTI intensity audit**, looking for:

- NaN, +Inf and −Inf in the actual preprocessed arrays.
- Exact counts of extreme absolute normalized values: `|z| > 6`, `10`, `30`, `100`, `1000`.
- Collapsed or nearly constant channels, per-channel mean/std/min/max and sampled quantiles.
- The four priority cases: `sub_stroke0016`, `sub_stroke0113`, `sub_stroke0137`, and `sub_stroke0152`.
- Frozen patient-wise fold coverage (149/149), five-channel order and normalizer identities.

### Scientific safeguards

This is **read-only** for images, preprocessed tensors, labels, splits, model checkpoints and all prior results. Every output is written into a **new** private Drive directory, `results/perfusion_preprocessed_tensor_audit/`. The script opens one tensor in read mode and scans small spatial slabs to limit RAM. It writes a patient-level checkpoint before proceeding; **rerun Run all after a Colab disconnect to resume**.

This is a *summary-level* raw-vs-preprocessed comparison, **not voxel-aligned** (nnU-Net crops and resamples before normalization). The stored Drive tensors may differ from any locally rebuilt preprocessing used during individual past training sessions; consequently this does not by itself prove what each historical GPU session read. No clipping rules or sanitization are applied or recommended automatically.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Frozen paths and pinned script

The code is fetched from a fixed Git commit, and the downloaded bytes are checked against the expected SHA256. The existing Notebook 17 raw-intensity CSV is a required input; no image data leave Google Drive.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, subprocess, sys

ROOT = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
DATASET = ROOT / 'nnUNet_preprocessed' / 'Dataset505_ISLES2024_NCCT_PERFUSION'
SPLITS = DATASET / 'splits_final.json'
RAW = ROOT / 'results' / 'perfusion_intensity_audit' / 'isles2024_perfusion_case_intensity_audit_PRIVATE.csv'
RAW_PROGRESS = ROOT / 'results' / 'perfusion_intensity_audit' / 'intensity_audit_progress.json'
OUT = ROOT / 'results' / 'perfusion_preprocessed_tensor_audit'
SCRIPT = Path('/content/audit_isles2024_preprocessed_tensors.py')
SCRIPT_COMMIT = 'eaef58d516ed156e32e4796c31bf7d04e320ecfd'
EXPECTED_SCRIPT_SHA256 = '54bea41511fcabb1ba7baa08d6d80e33231ef929ff3db67d27848367a217606a'
LOG = OUT / 'preprocessed_tensor_audit.log'

for p in (DATASET, DATASET / 'nnUNetPlans_3d_fullres', SPLITS, RAW, RAW_PROGRESS):
    if not p.exists(): raise FileNotFoundError(p)
previous = json.loads(RAW_PROGRESS.read_text(encoding='utf8'))
assert previous['status'] == 'complete' and previous['completed_cases'] == 149, previous
OUT.mkdir(parents=True, exist_ok=True)

def run_logged(label, cmd):
    cmd = [str(a) for a in cmd]
    print('\n===', label, '===', flush=True)
    with LOG.open('a', encoding='utf8') as stream:
        stream.write(f'\n[{datetime.now(timezone.utc).isoformat()}] {label}: $ {" ".join(cmd)}\n')
        stream.flush()
        with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, env={**os.environ,'PYTHONUNBUFFERED':'1'}) as task:
            for line in task.stdout:
                print(line, end='', flush=True)
                stream.write(line); stream.flush()
            code = task.wait()
        stream.write(f'[exit_code={code}]\n'); stream.flush()
    if code: raise RuntimeError(f'{label} failed with exit code {code}; see {LOG}')

run_logged('Download SHA-pinned audit script', ['curl','-fLsS','--retry','5','--retry-delay','4',
    f'https://raw.githubusercontent.com/NTinkicht/HybridStrokeSeg-Reproduction/{SCRIPT_COMMIT}/scripts/audit_isles2024_preprocessed_tensors.py',
    '-o',SCRIPT])
sha = hashlib.sha256(SCRIPT.read_bytes()).hexdigest()
assert sha == EXPECTED_SCRIPT_SHA256, f'Script hash mismatch: {sha} != {EXPECTED_SCRIPT_SHA256}'
print('Frozen code SHA256:', sha)
print('Audit outputs:', OUT)


## 2. Install the CPU-only Blosc2 reader and audit all 149 stored tensors

Each patient has a compressed `.b2nd`, a reference segmentation `.b2nd`, and metadata `.pkl`. The input is never overwritten, and the audit does not load entire five-channel volumes into RAM. Every patient is independently checkpointed. The full scan reads substantial existing data from Google Drive and can take time; disconnections are recoverable.


In [ ]:
run_logged('Install CPU-only tensor dependencies', [sys.executable,'-m','pip','install',
    '--disable-pip-version-check','--only-binary=:all:',
    'numpy>=1.26,<3','blosc2>=4.0,<5'])
run_logged('Audit Dataset505 preprocessed tensors (149 × 5)', [
    sys.executable,'-u',SCRIPT,
    '--preprocessed-dataset',DATASET,
    '--split-file',SPLITS,
    '--raw-audit-csv',RAW,
    '--output-dir',OUT,
    '--slices-per-block','8',
    '--retries','3'])


## 3. Verify completion and review the aggregate findings

**Do not infer that the old models definitely trained on the exact bytes in this Drive cache** if any historical sessions rebuilt preprocessing on a local disk. Classify preprocessing anomalies, then decide whether a new controlled experiment is justified. All per-patient outputs remain private.


In [ ]:
progress = json.loads((OUT/'preprocessed_tensor_audit_progress.json').read_text(encoding='utf8'))
summary = json.loads((OUT/'isles2024_preprocessed_tensor_audit_aggregate.json').read_text(encoding='utf8'))
assert progress['status'] == 'complete' and progress['completed_cases'] == 149, progress
assert summary['n_patients'] == 149 and summary['n_channel_volumes'] == 745, summary
assert summary['read_only_input'] is True
print('Verified complete:',summary['n_patients'],'patients × 5 channels')
print('Normalization:',summary['normalization_schemes'])
for channel, x in summary['channels'].items():
    print(f'{channel}: pre nonfinite {x["pre_cases_with_nonfinite"]}/149 '
          f'(total {x["pre_total_nonfinite"]}); |z|>100 {x["pre_cases_abs_gt_100"]}/149 '
          f'(total {x["pre_total_abs_gt_100"]}); near-constant {x["pre_cases_near_constant"]}')
print('PRIORITY CASES:')
for item in summary['priority_cases']:
    print(item)
print('Private CSV:',OUT/'isles2024_preprocessed_tensor_audit_PRIVATE.csv')
print('Aggregate:',OUT/'isles2024_preprocessed_tensor_audit_aggregate.json')
print('Progress:',OUT/'preprocessed_tensor_audit_progress.json')
